# Bearing Capacity with Direct LLM API Calls

### Learning objectives
- Verify a live call to the model with just a few lines of code
- Marshal deterministically validated soil/foundation data via `pydantic`
- Extract parameters using OpenAI structured outputs and cross-check with Terzaghi math
- Produce figures, sensitivity plots, and final JSON summaries ready for reports

In [ ]:
!pip install -q openai pydantic chromadb pypdf tiktoken python-dotenv

> **Colab note:** The `pip` cell above runs without modification in Google Colab; just make sure the repository (or at least `docs/07-llm/`) is uploaded so the PDFs can be read locally.

In [ ]:
import json
import math
import os
from pathlib import Path
from typing import List

import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv
from getpass import getpass
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError, computed_field, field_validator

## API key handling
1. Preferred: store `OPENAI_API_KEY` inside `.env` (never commit it).  
2. If the environment variable is missing (e.g., on Colab), the loader falls back to `getpass`.  
3. For unit tests, you can temporarily uncomment the placeholder line, but do not ship real keys in notebooks.

In [ ]:
def load_api_key(env_path: str = ".env") -> str:
    env_file = Path(env_path)
    if env_file.exists():
        load_dotenv(env_file)
    key = os.getenv("OPENAI_API_KEY")
    if not key:
        try:
            key = getpass("Enter your OpenAI API key: ").strip()
        except Exception as exc:
            raise RuntimeError("Unable to capture OPENAI_API_KEY interactively.") from exc
    if not key:
        raise RuntimeError("OPENAI_API_KEY is required. Set it in a .env file or type it interactively.")
    return key

# OPENAI_API_KEY = "sk-proj-example"  # Uncomment only for offline smoke tests
OPENAI_API_KEY = load_api_key()
print(f"Loaded API key prefix: {OPENAI_API_KEY[:8]}******** (masked)")
client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-5.4-mini"

## Quick sanity check: raw API call
Before diving into engineering workflows, confirm the connection by asking the model a trivial question.

In [ ]:
sanity = client.responses.create(
    model=MODEL,
    instructions="You are a succinct assistant.",
    input="Say hello to the AI in Geotech class in one sentence.",
)
print(sanity.output_text)
if sanity.usage:
    print("Token usage -> input:", sanity.usage.input_tokens, "output:", sanity.usage.output_tokens)

## Leaning Tower of Pisa soil narrative
The `gpt-bearing-capacity.pdf` report describes a layered clay profile supporting the circular masonry foundation (20 m diameter, embedded 5 m). We will feed that narrative to GPT for parameter extraction later.

In [ ]:
soil_report_text = '''
Estimate the site bearing capacity of a circular foundation with 2m diameter foundation with foundation resting on ground and undrained strength Su is 35 kPa
'''.strip()
print(soil_report_text)

## Simple model call (no Pydantic yet)
Ask the model for a bearing-capacity estimate. This is still a plain Responses API call: the system role goes in `instructions` and the question goes in `input`.

In [ ]:
tower_prompt = '''
Estimate the site bearing capacity of a circular foundation with 2m diameter foundation with foundation resting on ground and undrained strength Su is 35 kPa
'''.strip()


simple_completion = client.responses.create(
    model=MODEL,
    instructions="You are a meticulous geotechnical engineer.",
    input=tower_prompt,
)
print(simple_completion.output_text)
if simple_completion.usage:
    print("Token usage -> input:", simple_completion.usage.input_tokens,
          "output:", simple_completion.usage.output_tokens)